# Chapter 13 — Web Scraping

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 13](https://automatetheboringstuff.com/3e/chapter13.html)

**Setup note:** Needs `pip install requests beautifulsoup4`. To keep this notebook runnable offline, the example parses a local HTML string instead of hitting a live site — the BeautifulSoup API is identical either way.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 13.1 Fetching a page with requests

The third-party `requests` library is the standard way to download a web page in Python. `requests.get(url)` returns a Response object; `.status_code` (200 means OK) and `.text` (the raw HTML) are the two attributes you'll use constantly.


In [1]:
# import requests
# res = requests.get('https://automatetheboringstuff.com/')
# res.raise_for_status()   # raises an exception if the download failed
# html = res.text

# For an offline-runnable demo, we use a small HTML string instead:
html = '''
<html><body>
  <h1 id="title">Best Books</h1>
  <ul class="books">
    <li class="book">Automate the Boring Stuff</li>
    <li class="book">Fluent Python</li>
  </ul>
</body></html>
'''
print(html)



<html><body>
  <h1 id="title">Best Books</h1>
  <ul class="books">
    <li class="book">Automate the Boring Stuff</li>
    <li class="book">Fluent Python</li>
  </ul>
</body></html>



## 13.2 Parsing HTML with BeautifulSoup

`BeautifulSoup(html, 'html.parser')` turns raw HTML text into a searchable tree of elements. `.find()` gets the first matching element, `.find_all()` gets every match.


In [2]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, 'html.parser')
title = soup.find('h1')
print(title.text)

books = soup.find_all('li', class_='book')
for b in books:
    print('-', b.text)


Best Books
- Automate the Boring Stuff
- Fluent Python


## 13.3 Selecting elements with CSS selectors

`.select(css_selector)` lets you use familiar CSS selector syntax (`#id`, `.class`, `tag`) instead of `find`/`find_all` arguments — often faster to write once you know CSS.


In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html, 'html.parser')
print(soup.select('#title')[0].text)
for el in soup.select('.books .book'):
    print(el.get_text())


Best Books
Automate the Boring Stuff
Fluent Python


## 13.4 A note on JavaScript-rendered pages

`requests` only downloads the raw HTML that the server sends — it does not run JavaScript. If a page builds its content dynamically in the browser, `requests` + BeautifulSoup will see an empty shell. For those pages you need a browser-automation tool like Selenium or Playwright, which actually renders the page before you scrape it.


In [4]:
# Conceptual only — needs `pip install selenium` and a browser driver:
# from selenium import webdriver
# browser = webdriver.Chrome()
# browser.get('https://example.com')
# element = browser.find_element('css selector', '.some-class')
# print(element.text)
# browser.quit()
print('Selenium is the tool of choice when requests + BeautifulSoup see an empty page.')


Selenium is the tool of choice when requests + BeautifulSoup see an empty page.


## 13.5 Browser automation with Selenium

`requests` + `BeautifulSoup` only see a page's raw HTML — they can't click buttons, fill in forms, or wait for JavaScript to finish rendering content. `Selenium` drives a *real, actual browser* (Chrome, Firefox, etc.) under program control: `webdriver.Chrome()` launches the browser, `driver.get(url)` navigates, `driver.find_element(By.ID, 'search')` locates an element, `.click()` / `.send_keys('text')` interact with it exactly like a human would. This is the standard tool for scraping sites that build their content dynamically with JavaScript, or that require logging in / clicking through steps first.

**Environment note:** Selenium needs a real browser binary plus a matching driver (e.g. `chromedriver`) installed on the machine — neither is available in this headless sandbox (no display, no browser package, no root access to install one). The code below is real, correct Selenium API usage — copy it into Cursor on a machine with Chrome installed (`pip install selenium` handles the driver automatically in modern Selenium versions) to actually run it.

In [5]:
selenium_demo_code = '''
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

driver = webdriver.Chrome()          # launches a real, visible Chrome window
driver.get('https://www.python.org')

search_box = driver.find_element(By.NAME, 'q')   # locate the search input by its "name" attribute
search_box.send_keys('automate the boring stuff')
search_box.send_keys(Keys.RETURN)                 # press Enter, like a human would

results = driver.find_elements(By.CSS_SELECTOR, 'li.list-recent-events a')
for r in results[:5]:
    print(r.text)

driver.quit()
'''

print('This sandbox has no browser binary/driver installed, so this is illustrative only.')
print('Copy this into a Python 3.11+ environment with Chrome installed to run it for real:\n')
print(selenium_demo_code)


This sandbox has no browser binary/driver installed, so this is illustrative only.
Copy this into a Python 3.11+ environment with Chrome installed to run it for real:


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

driver = webdriver.Chrome()          # launches a real, visible Chrome window
driver.get('https://www.python.org')

search_box = driver.find_element(By.NAME, 'q')   # locate the search input by its "name" attribute
search_box.send_keys('automate the boring stuff')
search_box.send_keys(Keys.RETURN)                 # press Enter, like a human would

results = driver.find_elements(By.CSS_SELECTOR, 'li.list-recent-events a')
for r in results[:5]:
    print(r.text)

driver.quit()



## 13.6 Browser automation with Playwright

`Playwright` (originally from Microsoft) does the same job as Selenium — driving a real browser — with a more modern API and built-in auto-waiting (it waits for an element to actually be ready before interacting, instead of you writing manual `sleep()` calls). `sync_playwright()` gives you a browser/page object; `page.goto(url)` navigates, `page.fill(selector, text)` types into a field, `page.click(selector)` clicks. Playwright can also run **headless** (no visible window) by default, which is handy for automation running on a server.

**Environment note:** like Selenium, Playwright needs browser binaries. `playwright install chromium` downloads them, but that download also needs OS-level system libraries (`libxdamage1` and others) that require root to install — unavailable in this sandbox. Again, the code below is real and correct — it will run as-is on a machine where you've completed the one-time `pip install playwright && playwright install` setup.

In [6]:
playwright_demo_code = '''
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)   # no visible window; set headless=False to watch it work
    page = browser.new_page()
    page.goto('https://www.python.org')
    page.fill('input[name="q"]', 'automate the boring stuff')
    page.press('input[name="q"]', 'Enter')
    page.wait_for_selector('li.list-recent-events')   # auto-waits, no manual sleep() needed
    titles = page.eval_on_selector_all('li.list-recent-events a', 'els => els.map(e => e.textContent)')
    print(titles[:5])
    browser.close()
'''

print('This sandbox cannot download/install browser binaries (needs root for system deps).')
print('Copy this into your own environment after `pip install playwright && playwright install`:\n')
print(playwright_demo_code)


This sandbox cannot download/install browser binaries (needs root for system deps).
Copy this into your own environment after `pip install playwright && playwright install`:


from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)   # no visible window; set headless=False to watch it work
    page = browser.new_page()
    page.goto('https://www.python.org')
    page.fill('input[name="q"]', 'automate the boring stuff')
    page.press('input[name="q"]', 'Enter')
    page.wait_for_selector('li.list-recent-events')   # auto-waits, no manual sleep() needed
    titles = page.eval_on_selector_all('li.list-recent-events a', 'els => els.map(e => e.textContent)')
    print(titles[:5])
    browser.close()



## 13.7 Opening a page with the built-in webbrowser module

Not every scraping task needs a headless/automated browser — sometimes you just want your script to open a URL in the user's actual, already-configured browser (to show them a result, for instance). The standard library's `webbrowser.open(url)` does exactly that, with zero installs required. It returns `True` if it believes it launched a browser successfully, `False` otherwise — it does *not* let you read the page or interact with it (for that, you're back to Selenium/Playwright or `requests`).

In [7]:
import webbrowser

result = webbrowser.open('https://automatetheboringstuff.com/3e/')
print('webbrowser.open() reported success:', result)
print('(False here just means this headless sandbox has no browser registered to launch --')
print(' on a real desktop this actually opens the URL in your default browser.)')


webbrowser.open() reported success: False
(False here just means this headless sandbox has no browser registered to launch --
 on a real desktop this actually opens the URL in your default browser.)
